# trainer-class-skeleton — worked example 2: Trainer that stops early when validation loss stops improving

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `trainer-class-skeleton`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Early stopping prevents overfitting by halting training when the validation loss has not improved for a fixed number of epochs (the patience). The Trainer keeps track of the best validation loss seen so far and a counter of epochs without improvement. When the counter exceeds the patience, `fit` exits early. This pattern is an extension of the base skeleton that adds state (`best_val_loss`, `epochs_no_improve`) without touching the core loop structure.

## Worked solution

**Step 1 – Extra `__init__` state.** Beyond the base five attributes, we add `self.patience`, `self.best_val_loss = float('inf')`, and `self.epochs_no_improve = 0`.

**Step 2 – `fit` checks a stop flag.** After each epoch's `validate()` call, we inspect the last val loss. If it improved (strictly less than `best_val_loss`), we reset the counter. Otherwise we increment it. If `epochs_no_improve >= patience`, we break out of the epoch loop.

**Step 3 – Return early-stop info.** The method returns a dict with `'stopped_early'` (bool) and `'stop_epoch'` (which epoch triggered the stop, or `n_epochs` if training ran to completion). This lets callers know whether early stopping fired.

**Step 4 – No changes to `_step` or `validate`.** Those methods are inherited from the base pattern. Early stopping is purely a `fit`-level concern.

In [ ]:
import torch as t
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class WorkedTrainer2EarlyStop:
    def __init__(self, model, optimizer, train_loader, val_loader, loss_fn, patience=2):
        self.model = model
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.loss_fn = loss_fn
        self.patience = patience
        self.step = 0
        self.history = {'train_loss': [], 'val_loss': []}
        self.best_val_loss = float('inf')
        self.epochs_no_improve = 0

    def _step(self, x, y):
        return self.loss_fn(self.model(x), y)

    def fit(self, n_epochs):
        stopped_early = False
        stop_epoch = n_epochs
        for epoch in range(n_epochs):
            self.model.train()
            for x, y in self.train_loader:
                loss = self._step(x, y)
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()
                self.step += 1
                self.history['train_loss'].append(loss.item())
            self.validate()
            val_loss = self.history['val_loss'][-1]
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.epochs_no_improve = 0
            else:
                self.epochs_no_improve += 1
            if self.epochs_no_improve >= self.patience:
                stopped_early = True
                stop_epoch = epoch + 1
                break
        return {'stopped_early': stopped_early, 'stop_epoch': stop_epoch}

    def validate(self):
        self.model.eval()
        total, count = 0.0, 0
        with t.inference_mode():
            for x, y in self.val_loader:
                loss = self.loss_fn(self.model(x), y)
                total += loss.item() * x.shape[0]
                count += x.shape[0]
        self.history['val_loss'].append(total / count)

# Demo
t.manual_seed(3)
X = t.randn(60, 3)
Y = (X[:, 0] - X[:, 2]).unsqueeze(1)
train_dl = DataLoader(TensorDataset(X[:50], Y[:50]), batch_size=10)
val_dl = DataLoader(TensorDataset(X[50:], Y[50:]), batch_size=10)
model = nn.Linear(3, 1)
trainer = WorkedTrainer2EarlyStop(model, t.optim.SGD(model.parameters(), lr=0.1),
                                   train_dl, val_dl, nn.MSELoss(), patience=2)
info = trainer.fit(20)
print('early stop info:', info)
print('val_loss history:', trainer.history['val_loss'])